
# PatchTST (HF 사전학습 ckpt) → Fine-tuning → 예측

- `patchtst-etth1-pretrain/` 폴더의 **`pytorch_model.bin`**을 백본으로 로드한 뒤, 우리 데이터로 **linear probing → full finetuning** 수행.
- 입력 28일 → 예측 7일. 손실 MAE, 검증지표 sMAPE.
- `TEST_**.csv`에 대해 예측 생성.

> 주의: HuggingFace 형식의 키 네이밍이 원본 구현과 다를 수 있음. 본 노트북은 **백본만** 최대한 매핑해서 로드하고, 예측 head는 새로 학습한다.


In [29]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [30]:

# === 1) Config ===
from pathlib import Path
import os

# 경로
DATA_DIR = Path("./dataset")                 # 필요 시 변경
TRAIN_CSV = DATA_DIR / "train.csv"    # ex) ./train.csv
TEST_DIR  = DATA_DIR                  # TEST_00..*.csv 들이 있는 곳
HF_DIR    = Path("./patchtst-etth1-pretrain")   # HuggingFace ckpt 폴더
HF_BIN    = HF_DIR / "pytorch_model.bin"
SUBMISSION_OUT = "submission_patchtst_hf.csv"

# 모델 하이퍼파라미터 (ETT pretrain 기본값과 호환되도록 설정)
SEQ_LEN   = 28
PRED_LEN  = 7
PATCH_LEN = 14
STRIDE    = 1
DMODEL   = 256
NHEADS   = 8
FFN_DIM  = 1024
ELAYERS  = 4
DROPOUT  = 0.1

# 학습 설정
BATCH_SIZE           = 256
LR_LINEAR_PROBE      = 5e-4
LR_FINETUNE          = 1e-4
EPOCHS_LINEAR_PROBE = 0   # 바로 FT
EPOCHS_FINETUNE     = 40
LR_FINETUNE         = 3e-4
WEIGHT_DECAY        = 1e-2
EARLY_STOP_PATIENCE = 8
SEED = 42
NUM_WORKERS = 2

# 디바이스
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

assert TRAIN_CSV.exists(), f"not found: {TRAIN_CSV}"
assert HF_BIN.exists(), f"not found: {HF_BIN}"


device: cuda


In [31]:

# === 2) Imports & Utils ===
import math, random, warnings
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

def smape(y_true, y_pred, eps=1e-8):
    denom = (torch.abs(y_true) + torch.abs(y_pred)).clamp_min(eps)
    return (200.0 * torch.mean(torch.abs(y_pred - y_true) / denom))

class EarlyStopper:
    def __init__(self, patience=3, mode="min"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.count = 0
        self.should_stop = False

    def step(self, value):
        if self.best is None:
            self.best = value
            return False
        improved = (value < self.best) if self.mode == "min" else (value > self.best)
        if improved:
            self.best = value
            self.count = 0
            return False
        self.count += 1
        if self.count >= self.patience:
            self.should_stop = True
        return self.should_stop


In [32]:

# === 3) Load data ===
df = pd.read_csv(TRAIN_CSV)

# 표준 컬럼 보정
if "store_menu" not in df.columns:
    if "store_menu_id" in df.columns:
        df["store_menu"] = df["store_menu_id"]
    else:
        df["store_menu"] = df["store"].astype(str) + "_" + df["menu"].astype(str)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store_menu", "date"]).reset_index(drop=True)

# 음수 sales -> 0
if "sales" in df.columns:
    df.loc[df["sales"] < 0, "sales"] = 0

series_ids = df["store_menu"].unique().tolist()
print(f"Series: {len(series_ids)}, rows: {len(df)}, range: {df['date'].min()} ~ {df['date'].max()}")


Series: 193, rows: 102676, range: 2023-01-01 00:00:00 ~ 2024-06-15 00:00:00


In [33]:

# === 4) Dataset (28 -> 7) ===
class PanelWindowDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, seq_len: int, pred_len: int, val_cutoff_date=None):
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.X, self.Y, self.ids = [], [], []
        for sid, g in frame.groupby("store_menu"):
            g = g.sort_values("date").reset_index(drop=True)
            values = g["sales"].astype(float).values
            dates  = g["date"].values
            n = len(values)
            for s in range(0, n - (seq_len + pred_len) + 1):
                x = values[s : s + seq_len]
                y = values[s + seq_len : s + seq_len + pred_len]
                y_end_date = dates[s + seq_len + pred_len - 1]
                if val_cutoff_date is not None and y_end_date >= val_cutoff_date:
                    self.X.append(x); self.Y.append(y); self.ids.append(sid)
                elif val_cutoff_date is None:
                    self.X.append(x); self.Y.append(y); self.ids.append(sid)
        self.X = np.asarray(self.X, dtype=np.float32)
        self.Y = np.asarray(self.Y, dtype=np.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x = self.X[idx]; y = self.Y[idx]
        return torch.from_numpy(x[None, :]), torch.from_numpy(y), self.ids[idx]

last_date = df["date"].max()
val_cutoff = last_date - pd.Timedelta(days=35)

train_ds = PanelWindowDataset(df, SEQ_LEN, PRED_LEN, val_cutoff_date=None)
val_ds   = PanelWindowDataset(df, SEQ_LEN, PRED_LEN, val_cutoff_date=val_cutoff.to_datetime64())

from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

len(train_ds), len(val_ds)


(96114, 6948)

In [34]:

# === 5) PatchTST minimal backbone + head ===
class InstanceNormPerSample(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__(); self.eps = eps
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True).clamp_min(self.eps)
        return (x - mean) / std, mean, std

class PatchEmbed1D(nn.Module):
    def __init__(self, patch_len: int, stride: int):
        super().__init__(); self.patch_len = patch_len; self.stride = stride
    def forward(self, x):
        # x: (B,1,L)
        B, C, L = x.shape; P = self.patch_len; S = self.stride
        if L < P:
            x = F.pad(x, (0, P-L), mode="replicate"); L = P
        idx = torch.arange(0, max(L - P + 1, 1), S, device=x.device)
        last = L - P
        if idx.numel() == 0 or idx[-1].item() != last:
            idx = torch.cat([idx, torch.tensor([last], device=x.device)])
        patches = torch.stack([x[:, :, i:i+P] for i in idx], dim=2)  # (B,1,N,P)
        return patches.squeeze(1)  # (B,N,P)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        N = x.size(1); return x + self.pe[:, :N, :]

class PatchTSTBackbone(nn.Module):
    def __init__(self, patch_len=16, stride=8, d_model=128, n_heads=16, d_ff=256, e_layers=3, dropout=0.2):
        super().__init__()
        self.inorm  = InstanceNormPerSample()
        self.patcher= PatchEmbed1D(patch_len, stride)
        self.proj   = nn.Linear(patch_len, d_model)
        self.pos    = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder= nn.TransformerEncoder(enc_layer, num_layers=e_layers)

    def forward(self, x):
        x_norm, mean, std = self.inorm(x)
        tokens = self.patcher(x_norm)   # (B,N,P)
        z = self.proj(tokens)           # (B,N,D)
        z = self.pos(z)                 # (B,N,D)
        h = self.encoder(z)             # (B,N,D)
        return h, mean, std

class ForecastHead(nn.Module):
    def __init__(self, d_model=128, pred_len=7):
        super().__init__()
        self.pred_len = pred_len
        self.proj_in = None
        self.linear  = nn.Linear(d_model, pred_len)
    def forward(self, h, mean, std):
        B,N,D = h.shape
        if self.proj_in is None or self.proj_in.in_features != N*D:
            self.proj_in = nn.Linear(N*D, D).to(h.device)
        x = self.proj_in(h.reshape(B, N*D))
        x = F.relu(x)
        x = self.linear(x)             # (B, pred_len)
        x = x * std.squeeze(1) + mean.squeeze(1)
        return x

class PatchTSTForecast(nn.Module):
    def __init__(self, patch_len=16, stride=8, d_model=128, n_heads=16, d_ff=256, e_layers=3, dropout=0.2, pred_len=7):
        super().__init__()
        self.backbone = PatchTSTBackbone(patch_len, stride, d_model, n_heads, d_ff, e_layers, dropout)
        self.forecast_head = ForecastHead(d_model, pred_len)
    def forward(self, x):
        h, mean, std = self.backbone(x)
        return self.forecast_head(h, mean, std)


In [35]:
HF_BIN = "./patchtst-etth1-pretrain/pytorch_model.bin"

def assert_real_weights(path):
    import os
    sz = os.path.getsize(path)
    if sz < 1_000_000:  # 1MB 미만이면 의심
        with open(path, "rb") as f:
            head = f.read(64)
        if b"git-lfs" in head or head.startswith(b"version "):
            raise RuntimeError(
                "Git-LFS 포인터 파일입니다. 'git lfs pull' 또는 HF Hub로 실제 가중치를 다시 받으세요."
            )

def load_hf_state_dict(bin_path: str):
    assert_real_weights(bin_path)
    try:
        sd = torch.load(bin_path, map_location="cpu", weights_only=False)  # PyTorch 2.6 대응
    except TypeError:
        sd = torch.load(bin_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd and isinstance(sd["state_dict"], dict):
        sd = sd["state_dict"]
    if not isinstance(sd, dict):
        raise TypeError("Unexpected checkpoint format (not a state_dict).")
    return sd

def strip_prefix(k: str):
    for p in ["model.", "module.", "backbone.", "base_model.", "transformer.", "encoder."]:
        if k.startswith(p): return k[len(p):]
    return k

sd_raw = load_hf_state_dict(HF_BIN)
sd_mapped = {strip_prefix(k): v for k, v in sd_raw.items()}

res = model.backbone.load_state_dict(sd_mapped, strict=False)
print("Loaded HF ckpt to backbone with strict=False")
print("missing_keys (first 15):", res.missing_keys[:15])
print("unexpected_keys (first 15):", res.unexpected_keys[:15])


Loaded HF ckpt to backbone with strict=False
missing_keys (first 15): ['proj.weight', 'proj.bias', 'pos.pe', 'encoder.layers.0.self_attn.in_proj_weight', 'encoder.layers.0.self_attn.in_proj_bias', 'encoder.layers.0.self_attn.out_proj.weight', 'encoder.layers.0.self_attn.out_proj.bias', 'encoder.layers.0.linear1.weight', 'encoder.layers.0.linear1.bias', 'encoder.layers.0.linear2.weight', 'encoder.layers.0.linear2.bias', 'encoder.layers.0.norm1.weight', 'encoder.layers.0.norm1.bias', 'encoder.layers.0.norm2.weight', 'encoder.layers.0.norm2.bias']
unexpected_keys (first 15): ['head.linear.weight', 'head.linear.bias', 'encoder.w_pos', 'encoder.w_p.weight', 'encoder.w_p.bias', 'encoder.encoder.layers.0.self_attn.k_proj.weight', 'encoder.encoder.layers.0.self_attn.k_proj.bias', 'encoder.encoder.layers.0.self_attn.v_proj.weight', 'encoder.encoder.layers.0.self_attn.v_proj.bias', 'encoder.encoder.layers.0.self_attn.q_proj.weight', 'encoder.encoder.layers.0.self_attn.q_proj.bias', 'encoder.enco

In [38]:

# === 7) Train / Eval ===
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    loss_sum = 0.0; smape_sum = 0.0; n = 0
    for xb, yb, _ in loader:
        xb = xb.to(device); yb = yb.to(device)
        yp = model(xb)
        loss = F.l1_loss(yp, yb)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        with torch.no_grad():
            s = smape(yb, yp)
        bs = xb.size(0)
        loss_sum += loss.item()*bs; smape_sum += s.item()*bs; n += bs
    return loss_sum/n, smape_sum/n

def make_optimizer(params, lr, wd):
    return torch.optim.AdamW(params, lr=lr, weight_decay=wd)

# Linear probing
for p in model.backbone.parameters(): p.requires_grad = False
opt_lp = make_optimizer(model.forecast_head.parameters(), LR_LINEAR_PROBE, WEIGHT_DECAY)
es = EarlyStopper(patience=EARLY_STOP_PATIENCE, mode="min")
print("== Linear probing ==")
for ep in range(1, EPOCHS_LINEAR_PROBE+1):
    tr_loss, tr_smape = run_epoch(model, train_loader, opt_lp)
    vl_loss, vl_smape = run_epoch(model, val_loader, None)
    print(f"[LP][{ep:02d}] train_mae={tr_loss:.4f} sMAPE={tr_smape:.2f}  val_mae={vl_loss:.4f} sMAPE={vl_smape:.2f}")
    if es.step(vl_smape):
        print("Early stop LP."); break

# Finetune
for p in model.backbone.parameters(): p.requires_grad = True
opt_ft = make_optimizer(model.parameters(), LR_FINETUNE, WEIGHT_DECAY)

from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(opt_ft, T_max=EPOCHS_FINETUNE, eta_min=1e-6)  # eta_min은 필요시 조정

es = EarlyStopper(patience=EARLY_STOP_PATIENCE, mode="min")
print("== Finetuning ==")
for ep in range(1, EPOCHS_FINETUNE+1):
    tr_loss, tr_smape = run_epoch(model, train_loader, optimizer=opt_ft)
    vl_loss, vl_smape = run_epoch(model, val_loader, optimizer=None)
    metric = vl_smape
    print(f"[FT][{ep:02d}] train_mae={tr_loss:.4f} sMAPE={tr_smape:.2f}  "
          f"val_mae={vl_loss:.4f} sMAPE={vl_smape:.2f}  lr={scheduler.get_last_lr()[0]:.2e}")

    if es.step(metric):
        print("Early stop FT.")
        break

    scheduler.step()

# Save
torch.save(model.state_dict(), "patchtst_finetuned_from_hf.pth")
print("saved:", Path("patchtst_finetuned_from_hf.pth").resolve())


== Linear probing ==
== Finetuning ==


[FT][01] train_mae=6.0081 sMAPE=144.03  val_mae=4.8827 sMAPE=142.73  lr=3.00e-04
[FT][02] train_mae=5.9201 sMAPE=144.13  val_mae=4.7759 sMAPE=138.86  lr=3.00e-04
[FT][03] train_mae=5.8630 sMAPE=143.98  val_mae=4.6634 sMAPE=139.68  lr=2.98e-04
[FT][04] train_mae=5.7853 sMAPE=144.08  val_mae=4.6803 sMAPE=138.59  lr=2.96e-04
[FT][05] train_mae=5.7315 sMAPE=144.04  val_mae=4.6482 sMAPE=139.96  lr=2.93e-04
[FT][06] train_mae=5.6990 sMAPE=144.16  val_mae=4.6143 sMAPE=139.37  lr=2.89e-04
[FT][07] train_mae=5.6561 sMAPE=144.11  val_mae=4.5716 sMAPE=138.41  lr=2.84e-04
[FT][08] train_mae=5.6157 sMAPE=144.05  val_mae=4.5411 sMAPE=139.08  lr=2.78e-04
[FT][09] train_mae=5.5468 sMAPE=144.17  val_mae=4.5452 sMAPE=139.24  lr=2.71e-04
[FT][10] train_mae=5.5275 sMAPE=144.12  val_mae=4.5252 sMAPE=138.57  lr=2.64e-04
[FT][11] train_mae=5.4858 sMAPE=144.16  val_mae=4.5221 sMAPE=139.24  lr=2.56e-04
[FT][12] train_mae=5.4506 sMAPE=144.07  val_mae=4.4809 sMAPE=138.26  lr=2.48e-04
[FT][13] train_mae=5.4031 sM

In [40]:

# === 8) Inference for TEST_**.csv ===
import numpy as np
import pandas as pd
from pathlib import Path
import os

def infer_one_file(model, csv_path: Path):
    df_t = pd.read_csv(csv_path)
    if "store_menu" not in df_t.columns:
        if "store_menu_id" in df_t.columns:
            df_t["store_menu"] = df_t["store_menu_id"]
        else:
            df_t["store_menu"] = df_t["store"].astype(str) + "_" + df_t["menu"].astype(str)
    df_t["date"] = pd.to_datetime(df_t["date"])
    df_t = df_t.sort_values(["store_menu", "date"]).reset_index(drop=True)

    preds = {}
    for sid, g in df_t.groupby("store_menu"):
        g = g.sort_values("date")
        x = g["sales"].astype(float).values
        x = np.clip(x, 0, None)
        assert len(x) == SEQ_LEN, f"{sid}: expected {SEQ_LEN}, got {len(x)}"
        xb = torch.tensor(x, dtype=torch.float32)[None, None, :]  # (1,1,L)
        with torch.no_grad():
            yp = model(xb.to(device)).cpu().numpy().reshape(-1)
        preds[sid] = yp
    return preds

model.eval()
test_files = sorted([p for p in TEST_DIR.glob("TEST_*.csv") if p.is_file()])
print("Found:", [p.name for p in test_files])

rows = []
for tf in test_files:
    preds = infer_one_file(model, tf)
    for sid, arr in preds.items():
        rows.append({"test_file": tf.name, "store_menu": sid, **{f"day+{i+1}": float(arr[i]) for i in range(PRED_LEN)}})

out_path = Path(SUBMISSION_OUT) if isinstance(SUBMISSION_OUT, str) else SUBMISSION_OUT

sub_df = pd.DataFrame(rows).sort_values(["test_file", "store_menu"]).reset_index(drop=True)
sub_df.to_csv(SUBMISSION_OUT, index=False)

print("saved:", str(out_path.resolve()))
sub_df.head()

Found: ['TEST_00.csv', 'TEST_01.csv', 'TEST_02.csv', 'TEST_03.csv', 'TEST_04.csv', 'TEST_05.csv', 'TEST_06.csv', 'TEST_07.csv', 'TEST_08.csv', 'TEST_09.csv']
saved: /home/wonjun/Aimers/submission_patchtst_hf.csv


,test_file,store_menu,day+1,day+2,day+3,day+4,day+5,day+6,day+7
0,TEST_00.csv,느티나무 셀프BBQ_1인 수저세트,5.011717,1.316542,1.006295,2.949693,3.154101,6.429126,10.600303
1,TEST_00.csv,느티나무 셀프BBQ_BBQ55(단체),-3.971519,-2.482658,-2.445408,0.775944,10.514343,2.797815,-2.195213
2,TEST_00.csv,"느티나무 셀프BBQ_대여료 30,000원",4.808197,0.152069,1.099034,1.249054,1.727479,4.990856,11.112712
3,TEST_00.csv,"느티나무 셀프BBQ_대여료 60,000원",2.419866,0.664890,0.900558,0.918978,0.894932,1.736358,4.891575
4,TEST_00.csv,"느티나무 셀프BBQ_대여료 90,000원",0.550099,0.136966,-0.050960,0.075926,0.073206,0.221385,1.407317
